# Ejercicio 1: Filtrado y ordenación de datos

### Paso 1. Importa las librerías necesarias

In [ ]:
import pandas as pd
import numpy as np

### Paso 2. Importa el conjunto de datos desde este enlace y asígnalo a una variable llamada chipo.

[Descargar el conjunto de datos](https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv)

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv"
chipo = pd.read_csv(url, sep="\t")
chipo

### Paso 3. Nombre del producto de mayor precio

In [ ]:
# Convertimos los precios a números. item_price es el importe total de la línea del pedido.
chipo['item_price_num'] = chipo['item_price'].str.replace('$', '', regex=False).astype(float)
chipo['unit_price'] = chipo['item_price_num'] / chipo['quantity']

# Producto con el precio más alto por unidad.
chipo.loc[chipo['unit_price'].idxmax(), ['item_name', 'unit_price']]

### Paso 4. ¿Cuántos productos cuestan más de 10 $?

Contamos los productos distintos cuyo precio por unidad supera los 10 $.

In [ ]:
# Productos distintos cuyo precio por unidad supera los 10 $.
productos_mas_de_10 = (
    chipo.loc[chipo['unit_price'] > 10, ['item_name', 'unit_price']]
    .drop_duplicates('item_name')
    .sort_values('unit_price', ascending=False)
)
print('Número de productos distintos:', productos_mas_de_10['item_name'].nunique())
productos_mas_de_10

### Paso 4.1. ¿Cuántos pedidos incluyen un producto de más de 10 $? ¿Es lo mismo?

In [ ]:
# Pedidos que incluyen al menos un producto cuyo precio por unidad supera los 10 $.
pedidos_con_producto_mas_de_10 = chipo.loc[chipo['unit_price'] > 10, 'order_id'].nunique()
pedidos_con_producto_mas_de_10

### Paso 4.2. ¿Cuántos pedidos tienen un importe total superior a 10 $? ¿Es lo mismo?

In [ ]:
# Calculamos el importe total de cada pedido y contamos los que superan los 10 $.
totales_por_pedido = chipo.groupby('order_id')['item_price_num'].sum()
(totales_por_pedido > 10).sum()

### Paso 4.3. ¿En cuántas líneas de pedido se ha pagado más de 10 $ por un mismo producto? ¿Es lo mismo?

In [ ]:
# Líneas de pedido con un importe superior a 10 $; mostramos también cuántos pedidos distintos son.
lineas_mas_de_10 = chipo.loc[chipo['item_price_num'] > 10]
print('Líneas de pedido de más de 10 $:', len(lineas_mas_de_10))
print('Pedidos distintos con alguna línea de más de 10 $:', lineas_mas_de_10['order_id'].nunique())
lineas_mas_de_10[['order_id', 'quantity', 'item_name', 'item_price', 'item_price_num']].head()

### Paso 5. Calcula el precio por unidad y llámalo unit_price. Muestra solo item_name y unit_price.

In [ ]:
# Precio por unidad de cada producto; quitamos duplicados para obtener una lista compacta.
precios_por_producto = (
    chipo.assign(unit_price=chipo['item_price_num'] / chipo['quantity'])
    .loc[:, ['item_name', 'unit_price']]
    .drop_duplicates()
    .sort_values('item_name')
    .reset_index(drop=True)
)
precios_por_producto

### Paso 6. Ordena los productos por nombre.

In [ ]:
# Ordenamos alfabéticamente por nombre del producto.
precios_por_producto.sort_values('item_name').reset_index(drop=True)

### Paso 7. ¿Qué cantidad se pidió del producto más caro? Resuélvelo de dos formas.

In [ ]:
# Línea de pedido de mayor importe. Mostramos el resultado con idxmax() y sort_values().
indice_mas_caro = chipo['item_price_num'].idxmax()
print('Con idxmax():')
display(chipo.loc[[indice_mas_caro], ['item_name', 'quantity', 'item_price', 'item_price_num']])

print('Con sort_values():')
display(chipo.sort_values('item_price_num', ascending=False).head(1)[['item_name', 'quantity', 'item_price', 'item_price_num']])

### Paso 8. ¿Cuántas veces se pidió una Veggie Salad Bowl?

In [ ]:
# Número de unidades vendidas y número de pedidos distintos que incluyeron una Veggie Salad Bowl.
ensalada_vegetal = chipo.loc[chipo['item_name'] == 'Veggie Salad Bowl']
print('Unidades pedidas:', ensalada_vegetal['quantity'].sum())
print('Pedidos distintos:', ensalada_vegetal['order_id'].nunique())

### Paso 9. ¿Cuántas veces se pidió más de una Canned Soda?

In [ ]:
# Líneas de Canned Soda con cantidad superior a uno; contamos líneas, pedidos y latas.
varias_sodas = chipo.loc[(chipo['item_name'] == 'Canned Soda') & (chipo['quantity'] > 1)]
print('Líneas de pedido:', len(varias_sodas))
print('Pedidos distintos:', varias_sodas['order_id'].nunique())
print('Latas en esas líneas:', varias_sodas['quantity'].sum())
varias_sodas[['order_id', 'quantity', 'item_name']]